# Day 059 — Exercise 4: Retry with Exponential Backoff

Background jobs fail for transient reasons: network blips, temporary overload, a model loading delay. **Exponential backoff** retries with increasing delays so you don't hammer a struggling service.

Pattern: attempt 0 → fail → sleep `d`, attempt 1 → fail → sleep `2d`, attempt 2 → fail → sleep `4d`, …, attempt `max_retries` → raise if still failing.

In [ ]:
import time
from typing import Any, Callable


## Task

Implement `retry_with_backoff(fn, max_retries=3, base_delay=0.001)`:

- Call `fn()` with no arguments
- If it raises, sleep `base_delay * 2**attempt` and retry (up to `max_retries` times)
- Return the result of the first successful call
- If all attempts fail, re-raise the **last** exception

## Your Implementation

In [ ]:
def retry_with_backoff(fn: Callable, max_retries: int = 3, base_delay: float = 0.001) -> Any:
    """Call fn(). On exception, retry up to max_retries times with exponential backoff.

    Delay before retry N: base_delay * (2 ** N) seconds  (0.001, 0.002, 0.004, ...)
    Raises the last exception if all attempts fail.
    fn is called with no arguments.
    """
    # TODO: loop up to max_retries+1 times, sleep between failures
    raise NotImplementedError


In [ ]:
def retry_with_backoff(fn, max_retries=3, base_delay=0.001):
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            if attempt < max_retries:
                time.sleep(base_delay * (2 ** attempt))
    raise last_exc


## Automated checks

In [ ]:
score, total = 0, 4
try:
    # succeeds immediately
    result = retry_with_backoff(lambda: 99, max_retries=3, base_delay=0)
    assert result == 99, f"Expected 99, got {result}"
    score += 1; print("\u2705 succeeds immediately and returns result")

    # fails N times then succeeds
    calls = {"n": 0}
    def flaky():
        calls["n"] += 1
        if calls["n"] < 3:
            raise ValueError("not yet")
        return "ok"

    result2 = retry_with_backoff(flaky, max_retries=5, base_delay=0)
    assert result2 == "ok", f"Expected 'ok', got {result2}"
    assert calls["n"] == 3, f"Expected 3 calls, got {calls['n']}"
    score += 1; print("\u2705 retries until success")

    # always fails → raises last exception
    always_fail = lambda: (_ for _ in ()).throw(RuntimeError("always"))
    raised = False
    try:
        retry_with_backoff(always_fail, max_retries=2, base_delay=0)
    except RuntimeError as e:
        raised = True
        assert str(e) == "always", f"Got {e}"
    assert raised, "Should have raised RuntimeError"
    score += 1; print("\u2705 raises last exception when all retries fail")

    # total call count = max_retries + 1
    n_calls = {"c": 0}
    def count_calls(): n_calls["c"] += 1; raise ValueError()
    try:
        retry_with_backoff(count_calls, max_retries=3, base_delay=0)
    except ValueError:
        pass
    assert n_calls["c"] == 4, f"Expected 4 calls (1 + 3 retries), got {n_calls['c']}"
    score += 1; print("\u2705 calls fn exactly max_retries + 1 times on total failure")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def retry_with_backoff(fn, max_retries=3, base_delay=0.001):
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            if attempt < max_retries:
                time.sleep(base_delay * (2 ** attempt))
    raise last_exc
```

**Why `range(max_retries + 1)`?** The first call (attempt 0) is not a retry — it is the initial attempt. So `max_retries=3` means 1 initial + 3 retries = 4 total calls. `raise last_exc` re-raises the last exception after all attempts are exhausted.

</details>